# 1. Pandas basics: from a trade blotter to an analysis
An execution file contains repeated tickers, trades from different desks, and reference data that does not cover every security. Before calculating portfolio exposures, we need to know what a row means and how pandas matches rows.

We will use a small synthetic trade blotter to learn Series, DataFrames, label lookup, joins, and aggregation. Prices and trades are illustrative.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

## 1. A Series stores values with labels
A **Series** is a one-dimensional collection of values with an index, a dtype, and optionally a name. Without an explicit index, pandas uses sequential integer labels starting at zero.

A label is an identifier, not a promise about the row's current position.

In [ ]:
price_series = pd.Series([420.0, 185.0, 198.0], name='price')
display(price_series)
print('Index:', price_series.index, '| dtype:', price_series.dtype)
by_ticker = pd.Series({'MSFT': 420.0, 'AAPL': 185.0, 'JPM': 198.0}, name='price')
display(by_ticker)
print('AAPL price by label:', by_ticker.loc['AAPL'])

## 2. A DataFrame adds columns
A **DataFrame** is a two-dimensional table. Its columns are Series sharing a row index; different columns can have different dtypes.

Here **one row is one execution**, not one stock. A ticker can appear more than once. `trade_id` is our business identifier; the default integer index is just a set of row labels.

In [ ]:
trades = pd.DataFrame({
    'trade_id': ['T101', 'T102', 'T103', 'T104', 'T105', 'T106'],
    'ticker': ['MSFT', 'AAPL', 'MSFT', 'JPM', 'XOM', 'AAPL'],
    'desk': ['Core', 'Core', 'Tactical', 'Core', 'Tactical', 'Tactical'],
    'side': ['BUY', 'BUY', 'SELL', 'BUY', 'SELL', 'BUY'],
    'shares': [20, 60, 15, 80, 100, 25],
    'price': [420.0, 185.0, 422.0, 198.0, 112.0, 186.0],
})
display(trades)
print('Rows, columns:', trades.shape)
display(trades.dtypes)
print(type(trades['price']), type(trades[['price']]))

`trades['price']` selects a column as a Series; `trades[['price']]` keeps a one-column DataFrame. A dictionary of lists supplies the columns above; a list of dictionaries would supply individual rows.

We can calculate a new column with ordinary arithmetic. `notional` below is the positive dollar size of an execution, including sells; it is not a portfolio holding or profit.

In [ ]:
trades['notional'] = trades['shares'] * trades['price']
display(trades[['trade_id', 'ticker', 'side', 'notional']])
display(trades[['shares', 'price', 'notional']].describe().round(2))

## 3. The index trap: `.loc` is not `.iloc`
`.loc[label]` looks up an **index label**. `.iloc[position]` selects a **position**, counting from zero. With the original default index, `.loc[0]` and `.iloc[0]` select the same execution.

Sorting moves rows but keeps their labels. After sorting by notional, which execution will `.loc[0]` return? Which will `.iloc[0]` return?

In [ ]:
ranked = trades.sort_values('notional', ascending=False)
display(ranked[['trade_id', 'ticker', 'notional']])
print('Label 0 still identifies the original MSFT trade:')
display(ranked.loc[0, ['trade_id', 'ticker', 'notional']])
print('Position 0 is now the largest execution:')
display(ranked.iloc[0][['trade_id', 'ticker', 'notional']])
sorted_prices = price_series.sort_values()
print('Series: label 0 =', sorted_prices.loc[0], '| position 0 =', sorted_prices.iloc[0])

### Choose the index that answers the question
Use `.iloc[:2]` for the first two rows, regardless of labels. Use `set_index('trade_id')` when you want to retrieve a known execution. Label slices include the ending label; positional slices exclude the ending position.

`reset_index(drop=True)` assigns new sequential labels. Keep `trade_id` as a column so resetting the index does not erase the execution's identity. Filtering also retains labels, so the same distinction matters after a filter.

In [ ]:
display(ranked.iloc[:2][['trade_id', 'notional']])
by_trade = trades.set_index('trade_id')
display(by_trade.loc['T102':'T104', ['ticker', 'notional']])
display(by_trade.iloc[1:3][['ticker', 'notional']])
ranked_reset = ranked.reset_index(drop=True)
print('After resetting: label 0 now identifies', ranked_reset.loc[0, 'trade_id'])

### Arithmetic and assignment also match labels
The shares and prices below arrive in different orders. Multiplication matches ticker labels, so MSFT's shares receive MSFT's price. `pd.concat(..., axis=1)` also aligns columns by their indices.

An unmatched label produces a missing value. Converting to arrays would discard those labels and make matching depend on position.

In [ ]:
share_series = pd.Series({'JPM': 80, 'MSFT': 20, 'AAPL': 60}, name='shares')
aligned_values = share_series * by_ticker
display(pd.concat([share_series, by_ticker, aligned_values.rename('value')], axis=1))
by_trade['check_value'] = trades.set_index('trade_id')['notional'].sort_index(ascending=False)
assert by_trade['check_value'].equals(by_trade['notional'])

### Select and update with an explicit condition
A comparison returns one Boolean per row. Combine conditions with `&` (and), `|` (or), and parentheses. `.loc[rows, columns]` selects both dimensions.

Use `.copy()` before changing a selected table, and assign through `.loc` rather than a chain of selections.

In [ ]:
large_buys = trades.loc[(trades['side'] == 'BUY') & (trades['notional'] >= 10000),
                        ['trade_id', 'ticker', 'notional']]
display(large_buys)
corrected = trades.copy()
corrected.loc[corrected['trade_id'] == 'T103', 'shares'] = 18
corrected['notional'] = corrected['shares'] * corrected['price']
display(corrected.loc[corrected['trade_id'] == 'T103'])

## 4. Join types decide which securities survive
The blotter contains XOM, which is absent from this reference file. The reference file contains TSLA, which was not traded. These mismatches make the join choices visible.

| Join | Tickers retained | A useful question |
|---|---|---|
| `inner` | In both tables | Which traded names have reference data? |
| `left` | Every left-hand ticker | Can I enrich the traded universe without losing a name? |
| `right` | Every right-hand ticker | Which reference names appeared in the blotter? |
| `outer` | Either table | Where does coverage differ? |

Missing matches remain missing; they do not mean zero exposure.

In [ ]:
traded_names = trades[['ticker']].drop_duplicates()
security_master = pd.DataFrame({
    'ticker': ['AAPL', 'MSFT', 'JPM', 'TSLA'],
    'sector': ['Technology', 'Technology', 'Financials', 'Consumer'],
}, index=[101, 102, 103, 104])
display(traded_names)
display(security_master)

In [ ]:
join_results = {}
for how in ['inner', 'left', 'right', 'outer']:
    join_results[how] = traded_names.merge(security_master, on='ticker', how=how,
                                           validate='one_to_one', indicator=True)
    print(how.upper())
    display(join_results[how])

### Column keys and index keys are different choices
`merge(on='ticker')` matches the ticker column and ignores the original row indices. The result has a new integer index. The unrelated labels 101–104 are not security identifiers.

`join` matches indices by default. Set **both** indices to ticker before using it. With `join(on='ticker')`, the left ticker column matches the **right index**, and the left row labels are retained. The order of the reference rows does not have to match the trades.

In [ ]:
master_by_ticker = security_master.set_index('ticker').sort_index(ascending=False)
index_join = traded_names.set_index('ticker').join(master_by_ticker, how='left', validate='one_to_one')
column_to_index_join = traded_names.join(master_by_ticker, on='ticker', how='left', validate='one_to_one')
display(index_join)
display(column_to_index_join)
print('Column merge index:', join_results['left'].index.tolist())
print('Column-to-index join index:', column_to_index_join.index.tolist())

### A left join can still multiply rows
The full blotter has several executions per ticker. It needs a **many-to-one** join: many trades can match one reference row. If the reference file contains two AAPL rows, each AAPL execution would match twice.

`validate='many_to_one'` checks that assumption. Fix the reference conflict before calculating totals; dropping an arbitrary row could choose the wrong information.

In [ ]:
enriched = trades.merge(security_master, on='ticker', how='left', validate='many_to_one')
assert len(enriched) == len(trades)
display(enriched.loc[enriched['sector'].isna(), ['trade_id', 'ticker', 'notional']])
bad_master = pd.concat([security_master, security_master.loc[[101]]])
try:
    trades.merge(bad_master, on='ticker', how='left', validate='many_to_one')
except pd.errors.MergeError:
    print('Repeated key in the reference table: the join would multiply executions.')

### Appending rows is a different operation
`concat` stacks another delivery of executions. It keeps old index labels unless `ignore_index=True` is requested. The `trade_id` column still identifies the executions.

In [ ]:
late_trade = trades.iloc[[0]].copy()
late_trade.loc[:, 'trade_id'] = 'T107'
late_trade.loc[:, 'shares'] = 5
late_trade.loc[:, 'notional'] = late_trade['shares'] * late_trade['price']
stacked = pd.concat([trades, late_trade])
appended = pd.concat([trades, late_trade], ignore_index=True)
print('Retained labels:', stacked.index.tolist())
print('New sequential labels:', appended.index.tolist())
display(appended.tail(2))

## 5. Aggregate at the level of the question
A Series aggregation returns one value; a DataFrame aggregation normally works down each selected column. `groupby` performs those calculations separately for each group.

Count executions and distinct tickers separately: three executions do not necessarily represent three different securities. Named aggregations keep the output columns readable.

In [ ]:
print('Total traded notional:', trades['notional'].sum())
display(trades[['shares', 'notional']].agg(['sum', 'mean', 'max']))
desk_summary = trades.groupby('desk').agg(
    executions=('trade_id', 'size'),
    distinct_tickers=('ticker', 'nunique'),
    gross_notional=('notional', 'sum'),
)
display(desk_summary)
print('Group keys became the index:', desk_summary.index.tolist())
display(desk_summary.reset_index())

### Keep unclassified trades visible
`size` counts rows; `count` counts nonmissing values. A missing sector is a coverage gap. `dropna=False` keeps that group in the summary so its dollars do not disappear.

Use `as_index=False` when you want group keys to remain ordinary columns for a later join.

In [ ]:
print('Executions:', len(enriched), '| Classified:', enriched['sector'].count())
sector_activity = enriched.groupby('sector', dropna=False, as_index=False).agg(
    executions=('trade_id', 'size'), gross_notional=('notional', 'sum'))
display(sector_activity)
assert np.isclose(sector_activity['gross_notional'].sum(), trades['notional'].sum())

## From executions to holdings
You can now select rows intentionally, control which names survive a join, and summarize at the right level. The next notebook applies these operations to holdings across two books, where portfolio weights and cost basis require more than counting rows.

Next: [From pandas to portfolio decisions](3_pandas_to_portfolio.ipynb).

## Exercises
Use the examples above as patterns. The tasks below change the selection, reference coverage, or grouping question.

### Your turn: find large sells without confusing label and position
Sort the blotter by notional from smallest to largest. Store the sorted table in `your_ranked` and the first execution's `trade_id` in `your_first_trade` using positional lookup.

Then select sells of at least $10,000 into `your_sells`, keeping `trade_id`, `ticker`, and `notional`. Explain why `your_ranked.loc[0]` is not the smallest execution.

In [ ]:
your_ranked = None
your_first_trade = None
your_sells = None

In [ ]:
if your_ranked is None or your_sells is None:
    print('Sort the executions, select the first by position, then filter the large sells.')
else:
    pd.testing.assert_frame_equal(your_ranked, trades.sort_values('notional'))
    assert your_first_trade == 'T106'
    assert your_ranked.loc[0, 'trade_id'] == 'T101'
    assert your_sells['trade_id'].tolist() == ['T105']
    assert your_sells.columns.tolist() == ['trade_id', 'ticker', 'notional']
    print('Selection checks passed. Explain what stayed attached to each row during sorting.')

### Your turn: reconcile a different reference file
This delivery covers XOM but omits JPM, and it includes an untraded name. Join it to `traded_names` with an outer join and `indicator=True`, storing the result in `your_coverage`.

Store the sets of tickers with `left_only` and `right_only` matches in `your_missing_reference` and `your_untraded`. Preserve the source tables.

In [ ]:
practice_master = pd.DataFrame({'ticker': ['XOM', 'MSFT', 'AAPL', 'UNH'],
                                'sector': ['Energy', 'Technology', 'Technology', 'Healthcare']},
                               index=[8, 3, 7, 2])
your_coverage = None
your_missing_reference = None
your_untraded = None

In [ ]:
if your_coverage is None:
    print('Build the coverage audit and identify the two kinds of missing match.')
else:
    assert len(your_coverage) == 5
    assert set(your_coverage['ticker']) == {'AAPL', 'MSFT', 'JPM', 'XOM', 'UNH'}
    assert set(your_coverage.loc[your_coverage['_merge'] == 'left_only', 'ticker']) == {'JPM'}
    assert set(your_coverage.loc[your_coverage['_merge'] == 'right_only', 'ticker']) == {'UNH'}
    assert your_missing_reference == {'JPM'} and your_untraded == {'UNH'}
    print('Coverage checks passed. Decide which join you would use to enrich the blotter.')

### Your turn: summarize buy and sell activity separately
Build `your_side_summary` with one row per side and columns `executions`, `distinct_tickers`, and `gross_notional`; use `side` as the index.

**Stretch:** group by both desk and side. Explain why adding buy and sell notional measures activity rather than net exposure.

In [ ]:
your_side_summary = None

In [ ]:
if your_side_summary is None:
    print('Group by side, then count executions and tickers and sum notional.')
else:
    assert your_side_summary.index.name == 'side'
    assert your_side_summary.loc['BUY', 'executions'] == 4
    assert your_side_summary.loc['SELL', 'distinct_tickers'] == 2
    assert np.isclose(your_side_summary.loc['BUY', 'gross_notional'], 39990)
    assert np.isclose(your_side_summary.loc['SELL', 'gross_notional'], 17530)
    print('Aggregation checks passed. Describe the units of each output column.')